<a href="https://oscar-defelice.github.io">
<img src="../../img/image.png" height="125" align="right" />
</a>

# TP 03-A — Naive Bayes for Text Classification

**Course:** Natural Language Processing  
**Session:** 3 / 8  
**Duration:** ~60 min

---

## Objectives

By the end of this session you will be able to:

- Implement a Multinomial Naive Bayes classifier from scratch (counting + log-probabilities)
- Understand the role of Laplace smoothing and tune the hyperparameter α
- Use sklearn's `MultinomialNB` and reproduce your scratch results
- Inspect the most informative features per class
- Compare raw counts vs TF-IDF representations
- Produce a full evaluation: per-class F1, confusion matrix, error analysis

---

## Roadmap

| Part | Task | Deliverable |
|------|------|-------------|
| 1 | Data loading & preprocessing | Reuse Session 01 pipeline |
| 2 | Scratch NB implementation | `NaiveBayesClassifier` class |
| 3 | sklearn NB + smoothing sweep | Validation accuracy vs α plot |
| 4 | Feature analysis | Top-k words per class |
| 5 | BoW counts vs TF-IDF | Comparison table |
| 6 | Error analysis | Written analysis |

Each part ends with a **📝 Your analysis** cell.

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import re
import math
import warnings
from collections import Counter, defaultdict
from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
sns.set_style("darkgrid")
SEED = 42
np.random.seed(SEED)

AG_NEWS_LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}
print("Imports OK")

---

## Part 1 — Data loading & preprocessing

We reuse the same pipeline from Session 01.  
If you saved your split from that session, load it here.  
Otherwise, the cell below rebuilds it from scratch.

**Do not change the split** — we need identical train/val/test sets across all sessions.

In [ ]:
def preprocess_text(text: str) -> str:
    """Minimal 5-step text normalisation pipeline.

    Steps applied in order:
    1. Lowercase
    2. Remove HTML tags
    3. Replace non-alphanumeric characters with spaces
    4. Collapse multiple spaces
    5. Strip leading/trailing whitespace

    Parameters
    ----------
    text : str
        Raw input string.

    Returns
    -------
    str
        Normalised string.

    Examples
    --------
    >>> preprocess_text("Hello, World! <br/>")
    'hello world'
    """
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def load_ag_news(
    val_size: float = 0.1,
    seed: int = SEED,
) -> tuple[list[str], list[str], list[str], list[int], list[int], list[int]]:
    """Load AG News, preprocess, and split into train/val/test.

    Parameters
    ----------
    val_size : float
        Fraction of the original training split to use as validation.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    tuple
        (train_texts, val_texts, test_texts, train_labels, val_labels, test_labels)
        Labels are integers 0–3 (remapped from 1–4).
    """
    raw = load_dataset("ag_news")
    rng = np.random.default_rng(seed)

    train_texts_raw = [r["text"] for r in raw["train"]]
    train_labels_raw = [r["label"] for r in raw["train"]]
    test_texts = [preprocess_text(r["text"]) for r in raw["test"]]
    test_labels = [r["label"] for r in raw["test"]]

    n = len(train_texts_raw)
    idx = rng.permutation(n)
    split = int(n * (1 - val_size))
    train_idx, val_idx = idx[:split], idx[split:]

    train_texts  = [preprocess_text(train_texts_raw[i]) for i in train_idx]
    train_labels = [train_labels_raw[i]                 for i in train_idx]
    val_texts    = [preprocess_text(train_texts_raw[i]) for i in val_idx]
    val_labels   = [train_labels_raw[i]                 for i in val_idx]

    return train_texts, val_texts, test_texts, train_labels, val_labels, test_labels


train_texts, val_texts, test_texts, train_labels, val_labels, test_labels = load_ag_news()
print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")
print(f"Train class distribution: {Counter(train_labels)}")

---

## Part 2 — Naive Bayes from scratch

Before using sklearn, implement NB manually. This forces you to understand every step.

### 2.1 — The `NaiveBayesClassifier` class

Implement a Multinomial Naive Bayes classifier that:
- Tokenises on whitespace (simple split)
- Estimates P(c) from label counts
- Estimates P(w | c) with Laplace smoothing
- Classifies using log-probabilities

**Design constraints:**
- Use numpy-style docstrings on every method
- Store log-probabilities (not raw probabilities) in `self.log_likelihoods`
- `predict` returns a numpy array of integer labels
- `predict_proba` returns normalised log-sums (optional, Part 2.3)

In [ ]:
class NaiveBayesClassifier:
    """Multinomial Naive Bayes text classifier with Laplace smoothing.

    The model represents documents as bags of whitespace-tokenised words.
    Training is closed-form: estimate priors from label counts and likelihoods
    from per-class word frequencies. Inference uses log-probabilities to avoid
    numerical underflow.

    Parameters
    ----------
    alpha : float
        Laplace smoothing parameter. Must be > 0. Default is 1.0.

    Attributes
    ----------
    classes_ : list[int]
        Sorted list of unique class labels seen during fit.
    log_priors_ : dict[int, float]
        log P(c) for each class c.
    log_likelihoods_ : dict[int, dict[str, float]]
        log P(w | c) for each class c and word w.
    vocab_ : set[str]
        Vocabulary seen during training.

    Examples
    --------
    >>> clf = NaiveBayesClassifier(alpha=1.0)
    >>> clf.fit(["goal scored penalty", "parliament senate vote"], [0, 1])
    >>> clf.predict(["goal penalty kick"])
    array([0])
    """

    def __init__(self, alpha: float = 1.0) -> None:
        self.alpha = alpha
        self.classes_: list[int] = []
        self.log_priors_: dict[int, float] = {}
        self.log_likelihoods_: dict[int, dict[str, float]] = {}
        self.vocab_: set[str] = set()

    def fit(self, texts: list[str], labels: list[int]) -> "NaiveBayesClassifier":
        """Estimate priors and likelihoods from training data.

        Parameters
        ----------
        texts : list[str]
            Pre-tokenised (whitespace-split) training documents.
        labels : list[int]
            Integer class labels, one per document.

        Returns
        -------
        NaiveBayesClassifier
            Fitted classifier (self), to allow method chaining.
        """
        # YOUR CODE HERE
        # Hint: build self.classes_, self.log_priors_, self.log_likelihoods_, self.vocab_
        raise NotImplementedError

    def _score(self, text: str) -> dict[int, float]:
        """Compute log P(c) + sum log P(w | c) for each class.

        Words not in vocabulary receive the smoothed out-of-vocabulary probability:
        log( alpha / (total_class_words + alpha * vocab_size) ).

        Parameters
        ----------
        text : str
            A single whitespace-tokenised document.

        Returns
        -------
        dict[int, float]
            Mapping from class label to log-score.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def predict(self, texts: list[str]) -> np.ndarray:
        """Predict the most probable class for each document.

        Parameters
        ----------
        texts : list[str]
            Documents to classify.

        Returns
        -------
        np.ndarray of shape (len(texts),)
            Predicted integer class labels.
        """
        # YOUR CODE HERE
        raise NotImplementedError

    def top_features(
        self, class_label: int, n: int = 10
    ) -> list[tuple[str, float]]:
        """Return the n words with the highest log-likelihood for a given class.

        Parameters
        ----------
        class_label : int
            Class to inspect.
        n : int
            Number of top features to return.

        Returns
        -------
        list[tuple[str, float]]
            List of (word, log_P(w | c)) pairs, sorted descending by score.
        """
        # YOUR CODE HERE
        raise NotImplementedError

### 2.2 — Sanity-check on a toy dataset

Before running on AG News, verify your implementation on a tiny example where you can compute the answer by hand.

In [ ]:
# ── Toy corpus ────────────────────────────────────────────────────────────────
toy_train = [
    "goal scored penalty kick offside",   # class 0 — Sports
    "goal championship final winner cup", # class 0 — Sports
    "parliament senate vote bill minister", # class 1 — Politics
    "election president democracy party",   # class 1 — Politics
]
toy_labels = [0, 0, 1, 1]

clf_toy = NaiveBayesClassifier(alpha=1.0)
clf_toy.fit(toy_train, toy_labels)

# Expected: class 0
pred = clf_toy.predict(["goal penalty kick"])
assert pred[0] == 0, f"Expected 0, got {pred[0]}"

# Expected: class 1
pred2 = clf_toy.predict(["parliament vote election"])
assert pred2[0] == 1, f"Expected 1, got {pred2[0]}"

print("Toy assertions passed ✓")
print(f"Top 3 words for Sports: {clf_toy.top_features(0, n=3)}")
print(f"Top 3 words for Politics: {clf_toy.top_features(1, n=3)}")

### 2.3 — Train on AG News and evaluate

In [ ]:
def evaluate_classifier(
    y_true: list[int],
    y_pred: np.ndarray,
    label_names: list[str],
    title: str = "Evaluation",
) -> dict[str, float]:
    """Print classification report and plot confusion matrix.

    Parameters
    ----------
    y_true : list[int]
        Ground-truth labels.
    y_pred : np.ndarray
        Predicted labels.
    label_names : list[str]
        Human-readable class names in label order.
    title : str
        Title for the confusion matrix figure.

    Returns
    -------
    dict[str, float]
        Dictionary with 'accuracy' and 'macro_f1'.
    """
    report = classification_report(
        y_true, y_pred, target_names=label_names, output_dict=True
    )
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    print(classification_report(y_true, y_pred, target_names=label_names))

    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

    return {
        "accuracy": report["accuracy"],
        "macro_f1": report["macro avg"]["f1-score"],
    }


# ── Train and evaluate ────────────────────────────────────────────────────────
clf_scratch = NaiveBayesClassifier(alpha=1.0)
clf_scratch.fit(train_texts, train_labels)

val_pred_scratch = clf_scratch.predict(val_texts)
label_names = [AG_NEWS_LABELS[i + 1] for i in range(4)]

results_scratch = evaluate_classifier(
    val_labels, val_pred_scratch, label_names,
    title="Scratch NB — validation",
)

📝 **Your analysis (2.3)**

1. What macro F1 did your scratch NB achieve on validation?
2. Which class has the lowest F1? Why might that be? Look at the confusion matrix for clues.
3. Does your implementation agree with sklearn's result below (Part 3)? If not, why?

*Your answer here.*

---

## Part 3 — sklearn Multinomial NB + smoothing sweep

### 3.1 — Reproduce the scratch result with sklearn

Use `CountVectorizer` + `MultinomialNB`. Verify that the results are close to your scratch implementation.

In [ ]:
def build_nb_pipeline(
    alpha: float = 1.0,
    vectorizer: str = "count",
    max_features: int | None = 50_000,
) -> Pipeline:
    """Build a sklearn Pipeline: vectoriser → MultinomialNB.

    Parameters
    ----------
    alpha : float
        Laplace smoothing parameter for MultinomialNB.
    vectorizer : {'count', 'tfidf'}
        Which vectoriser to use.
    max_features : int or None
        Maximum vocabulary size. None means no limit.

    Returns
    -------
    Pipeline
        Unfitted sklearn Pipeline.

    Raises
    ------
    ValueError
        If vectorizer is not 'count' or 'tfidf'.
    """
    # YOUR CODE HERE
    raise NotImplementedError


pipe_nb = build_nb_pipeline(alpha=1.0, vectorizer="count")
pipe_nb.fit(train_texts, train_labels)
val_pred_nb = pipe_nb.predict(val_texts)

results_nb = evaluate_classifier(
    val_labels, val_pred_nb, label_names,
    title="sklearn NB (count, α=1.0) — validation",
)
print(f"Scratch NB macro F1: {results_scratch['macro_f1']:.4f}")
print(f"sklearn NB macro F1: {results_nb['macro_f1']:.4f}")

### 3.2 — Smoothing hyperparameter sweep

The Laplace smoothing parameter α controls how much probability mass we give to unseen words.
Tune it on the **validation set**.

In [ ]:
def smoothing_sweep(
    alphas: Iterable[float],
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
) -> pd.DataFrame:
    """Evaluate MultinomialNB for a range of smoothing values.

    Parameters
    ----------
    alphas : Iterable[float]
        Values of α to evaluate.
    train_texts : list[str]
        Training documents.
    train_labels : list[int]
        Training labels.
    val_texts : list[str]
        Validation documents.
    val_labels : list[int]
        Validation labels.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['alpha', 'val_accuracy', 'val_macro_f1'].
    """
    # YOUR CODE HERE
    raise NotImplementedError


alphas = [0.001, 0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
sweep_df = smoothing_sweep(alphas, train_texts, train_labels, val_texts, val_labels)

# ── Plot the results ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sweep_df["alpha"], sweep_df["val_macro_f1"], marker="o", label="macro F1")
ax.semilogx(sweep_df["alpha"], sweep_df["val_accuracy"], marker="s", linestyle="--", label="accuracy")
ax.set_xlabel("α (smoothing)")
ax.set_ylabel("validation score")
ax.set_title("MultinomialNB: smoothing sweep")
ax.legend()
plt.tight_layout()
plt.show()

best_alpha = sweep_df.loc[sweep_df["val_macro_f1"].idxmax(), "alpha"]
print(f"Best α = {best_alpha}")
print(sweep_df.to_string(index=False))

📝 **Your analysis (3.2)**

1. What value of α gives the best validation macro F1?
2. What happens for very small α (e.g. 0.001)? Why?
3. What happens for very large α (e.g. 10)? Think about what smoothing does to the probability estimates.

*Your answer here.*

---

## Part 4 — Feature analysis: most informative words per class

One strength of Naive Bayes is interpretability. After training, you can inspect exactly which words
the model associates with each class.

For each class c, the most informative words are those maximising:

$$\text{score}(w, c) = \log P(w | c) - \frac{1}{|C|} \sum_{c'} \log P(w | c')$$

This is the log-likelihood of the word in class c minus the average log-likelihood across all classes.
A high score means the word is much more likely in class c than in other classes.

In [ ]:
def most_informative_features(
    pipeline: Pipeline,
    label_names: list[str],
    n: int = 20,
) -> dict[str, pd.DataFrame]:
    """Extract the n most informative features per class from a fitted NB pipeline.

    The informativeness score for word w in class c is:
    log P(w | c) minus the mean log P(w | c') over all classes c'.

    Parameters
    ----------
    pipeline : Pipeline
        Fitted sklearn Pipeline with a vectoriser as step 0 and MultinomialNB as step 1.
    label_names : list[str]
        Human-readable class names, in label index order.
    n : int
        Number of top features to return per class.

    Returns
    -------
    dict[str, pd.DataFrame]
        Mapping class_name → DataFrame with columns ['word', 'log_p', 'score'].
    """
    # YOUR CODE HERE
    # Hint: pipeline[0].get_feature_names_out() gives the vocabulary
    # pipeline[1].feature_log_prob_ has shape (n_classes, vocab_size)
    raise NotImplementedError


def plot_informative_features(
    features: dict[str, pd.DataFrame],
    n_plot: int = 15,
) -> None:
    """Bar plot of the most informative features per class.

    Parameters
    ----------
    features : dict[str, pd.DataFrame]
        Output of `most_informative_features`.
    n_plot : int
        Number of features to show per class.
    """
    n_classes = len(features)
    fig, axes = plt.subplots(1, n_classes, figsize=(4.5 * n_classes, 5))
    palette = sns.color_palette("muted", n_classes)

    for ax, (name, df), colour in zip(axes, features.items(), palette):
        top = df.head(n_plot)
        ax.barh(top["word"][::-1], top["score"][::-1], color=colour)
        ax.set_title(name, fontweight="bold")
        ax.set_xlabel("informativeness score")
        ax.tick_params(axis="y", labelsize=8)

    plt.suptitle("Most informative features per class (NB)", y=1.01)
    plt.tight_layout()
    plt.show()


# ── Use the best α from the sweep ─────────────────────────────────────────────
pipe_best = build_nb_pipeline(alpha=best_alpha, vectorizer="count")
pipe_best.fit(train_texts, train_labels)

features = most_informative_features(pipe_best, label_names, n=20)
for name, df in features.items():
    print(f"\n{name} — top 10:")
    print(df.head(10).to_string(index=False))

plot_informative_features(features, n_plot=15)

📝 **Your analysis (4)**

1. Do the top features per class make intuitive sense? Give two examples of expected and one example of surprising features.
2. Are there any words that appear in the top features of multiple classes? What does that suggest?
3. Would you expect TF-IDF features to produce different top words? Why?

*Your answer here.*

---

## Part 5 — BoW counts vs TF-IDF

We saw in Session 01 that TF-IDF downweights words that appear frequently across all documents.
Does this help Naive Bayes?

> **Note:** `MultinomialNB` requires non-negative features. Raw TF-IDF values are non-negative by construction,
> so this is fine.

### 5.1 — Head-to-head comparison

In [ ]:
def compare_vectorizers(
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
    alpha: float = 1.0,
) -> pd.DataFrame:
    """Compare CountVectorizer vs TfidfVectorizer for MultinomialNB.

    Parameters
    ----------
    train_texts : list[str]
        Training documents.
    train_labels : list[int]
        Training labels.
    val_texts : list[str]
        Validation documents.
    val_labels : list[int]
        Validation labels.
    alpha : float
        Smoothing parameter for MultinomialNB.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['vectorizer', 'val_accuracy', 'val_macro_f1'].
    """
    # YOUR CODE HERE
    raise NotImplementedError


comparison_df = compare_vectorizers(
    train_texts, train_labels, val_texts, val_labels, alpha=best_alpha
)
print(comparison_df.to_string(index=False))

### 5.2 — Add bigrams

Extend the feature space with unigrams + bigrams (`ngram_range=(1, 2)`).
Does capturing two-word phrases help?

In [ ]:
def ngram_sweep(
    train_texts: list[str],
    train_labels: list[int],
    val_texts: list[str],
    val_labels: list[int],
    alpha: float = 1.0,
) -> pd.DataFrame:
    """Compare unigram-only vs unigram+bigram features.

    Parameters
    ----------
    train_texts : list[str]
        Training documents.
    train_labels : list[int]
        Training labels.
    val_texts : list[str]
        Validation documents.
    val_labels : list[int]
        Validation labels.
    alpha : float
        Smoothing parameter for MultinomialNB.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['features', 'vocab_size', 'val_accuracy', 'val_macro_f1'].
    """
    # YOUR CODE HERE
    raise NotImplementedError


ngram_df = ngram_sweep(train_texts, train_labels, val_texts, val_labels, alpha=best_alpha)
print(ngram_df.to_string(index=False))

📝 **Your analysis (5)**

1. Does TF-IDF improve over raw counts? By how much? Why or why not?
2. Do bigrams help? By how much? What is the cost (vocabulary size)?
3. Which configuration is your final best NB model? Record its test set performance below.

*Your answer here.*

---

## Part 6 — Final evaluation on the test set

Run your best NB configuration on the test set (only once — no peeking to tune).

In [ ]:
# ── Select your best pipeline configuration here ──────────────────────────────
# YOUR CODE HERE — train the best config, predict on test_texts
# test_pred = ...

results_test = evaluate_classifier(
    test_labels, test_pred, label_names,  # noqa: F821
    title="Best NB — test set (final)",
)
print(f"\nTest accuracy:  {results_test['accuracy']:.4f}")
print(f"Test macro F1:  {results_test['macro_f1']:.4f}")

---

## Part 7 — Error analysis

Understanding where the model fails is as important as the metric.
Sample 10 misclassified examples from the validation set and analyse them.

In [ ]:
def sample_errors(
    texts: list[str],
    y_true: list[int],
    y_pred: np.ndarray,
    label_names: list[str],
    n: int = 10,
    seed: int = SEED,
) -> pd.DataFrame:
    """Sample n misclassified documents for qualitative analysis.

    Parameters
    ----------
    texts : list[str]
        Input documents.
    y_true : list[int]
        Ground-truth labels.
    y_pred : np.ndarray
        Predicted labels.
    label_names : list[str]
        Human-readable class names.
    n : int
        Number of errors to sample.
    seed : int
        Random seed.

    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['text', 'true_label', 'predicted_label'].
    """
    # YOUR CODE HERE
    raise NotImplementedError


errors = sample_errors(val_texts, val_labels, val_pred_nb, label_names)
pd.set_option("display.max_colwidth", 120)
print(errors.to_string(index=False))

📝 **Your analysis (7)**

1. What are the most common error patterns? (e.g. Business ↔ Sci/Tech confusion)
2. Pick two specific misclassified examples and explain why the model got them wrong from a BoW perspective.
3. What feature engineering change could fix one of these errors?

*Your answer here.*

---

## Summary — Results table

Fill in after completing all parts. This table feeds into the running course comparison.

In [ ]:
results_table = pd.DataFrame([
    {"model": "Majority baseline",         "features": "none",      "val_macro_f1": 0.25,  "test_macro_f1": "?"},
    {"model": "Scratch NB",                 "features": "BoW count", "val_macro_f1": results_scratch["macro_f1"], "test_macro_f1": "?"},
    {"model": "sklearn NB (α=1)",           "features": "BoW count", "val_macro_f1": results_nb["macro_f1"],     "test_macro_f1": "?"},
    # Add best NB, TF-IDF NB, bigram NB here
])
print(results_table.to_string(index=False))